<a href="https://colab.research.google.com/github/prangancode/lane-detection-using-semantic-models-for-bd-complex-road-scenarios/blob/main/Utilities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount Google Drive

This section connects our Google Colab notebook with our Google Drive. By mounting the Drive, we can seamlessly access datasets, save model checkpoints, and store logs for persistent usage.

### Why Mount Google Drive?
- 📂 **Access Large Datasets**: Directly use datasets stored in your Drive.
- 💾 **Save Model Checkpoints**: Save training progress to reload later.
- 📝 **Persistent Storage**: Keep results and logs available after the session ends.

### What Happens Here?
1. **Authentication**: we'll be prompted to authenticate with your Google account.
2. **Drive Mounting**: Drive will be mounted at `/content/drive`.
3. **File Access**: All files and directories in your Drive become accessible within this environment.

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# Binary mask creation

In this section, we create binary masks for **lane lines** from the COCO annotation JSON files. These binary masks will be used for training, validation, and testing the lane detection model.

### What’s Happening Here?
1. **Path Setup**:
   - Set up paths for COCO annotation JSON files (generated using Roboflow after manually annotating lane lines).
   - Define output folders where the binary masks will be saved.

2. **COCO Annotations**:
   - The annotation JSON files contain image details, including their dimensions and the segmentation data for lane lines.

3. **Processing Steps**:
   - **Extract Annotations**: Read the JSON files and extract segmentation data.
   - **Create Binary Masks**: For each image, generate a binary mask where:
     - `1` (white) represents lane lines.
     - `0` (black) represents the background.
   - **Save Masks**: Save the binary masks as `.jpg` files in the specified output folders.


### Subsections
- **Install Necessary Libraries**: Install `pycocotools` and `opencv-python` for handling COCO annotations and image processing.
- **Paths for Training Annotations and Masks**: Process COCO JSON files for training images and generate binary masks.
- **Paths for Testing Annotations and Masks**: Similarly, process COCO JSON files for test images.
- **Paths for Validation Annotations and Masks**: Repeat the process for validation images.

## Install necessary libraries

We use the following libraries:
- **`pycocotools`**: For parsing COCO annotation files.
- **`opencv-python`**: For image processing tasks like creating and saving binary masks.





In [ ]:
pip install pycocotools opencv-python

## Paths for training annotation and output folder




In [ ]:
# Paths for training annotation and output folder
annotation_file = "/content/drive/MyDrive/Object_Detection/traffic_sign_object_detection/Dataset/Roboflow_lane_annotation/train_annotation_coco/_annotations.coco.json"  # Path to your JSON file
output_folder = "/content/drive/MyDrive/Object_Detection/traffic_sign_object_detection/Dataset/masks/roboflow_train_truth_masks"  # Path to save binary masks
os.makedirs(output_folder, exist_ok=True)

## Train mask creation


```markdown
## ✏️ Train Mask Creation
This step processes the training COCO JSON file to create binary masks for the annotated lane lines.

### Process:
1. Load the COCO JSON file and extract image details and annotations.
2. For each image:
   - Create a blank binary mask (`0` for background).
   - Draw polygons on the mask based on segmentation data from the JSON.
   - Save the binary mask in the specified output folder.


# Load and process COCO annotations
with open(annotation_file, "r") as f:
    coco_data = json.load(f)

images = {img["id"]: img for img in coco_data["images"]}
annotations = coco_data["annotations"]

for image_id, image_info in images.items():
    # Code for creating binary masks (as shown earlier).
    ...


In [ ]:
import os
import json
import numpy as np
import cv2

# Load COCO annotations
with open(annotation_file, "r") as f:
    coco_data = json.load(f)

# Extract images and annotations
images = {img["id"]: img for img in coco_data["images"]}
annotations = coco_data["annotations"]

# Process each annotation
for image_id, image_info in images.items():
    # Get image details
    image_name = image_info["file_name"]
    height, width = image_info["height"], image_info["width"]
    print(f"Processing image: {image_name}, dimensions: ({height}, {width})")

    # Modify the image name to remove everything from the last underscore to the dot
    cleaned_name = image_name.split('_')[0] + ".jpg"

    # Create an empty binary mask
    binary_mask = np.zeros((height, width), dtype=np.uint8)

    # Find annotations for this image
    image_annotations = [ann for ann in annotations if ann["image_id"] == image_id]

    # Process each annotation for this image
    for annotation in image_annotations:
        segmentation = annotation["segmentation"]

        # Handle polygons
        if isinstance(segmentation, list):  # Polygon format
            for poly in segmentation:
                points = np.array(poly).reshape(-1, 2).astype(np.int32)
                print(f"Polygon points: {points}")
                cv2.fillPoly(binary_mask, [points], color=1)  # Fill the polygon with '1'

    # Save the binary mask
    output_mask_path = os.path.join(output_folder, cleaned_name)
    cv2.imwrite(output_mask_path, binary_mask * 255)  # Save as binary JPEG (scaled to 255 for visibility)
    print(f"Saved binary mask to: {output_mask_path}")


Processing image: train7091_jpg.rf.5fb31c1ca2205c1632b4bb595d323e23.jpg, dimensions: (1080, 1920)
Polygon points: [[ 404 1075]
 [ 440 1073]
 [ 525  937]
 [ 597  832]
 [ 601  821]
 [ 601  802]
 [ 595  801]
 [ 549  862]
 [ 408 1062]
 [ 401 1073]
 [ 404 1075]]
Polygon points: [[1038  700]
 [1113  758]
 [1154  784]
 [1203  827]
 [1258  868]
 [1262  875]
 [1289  892]
 [1294  900]
 [1300  900]
 [1307  909]
 [1332  926]
 [1338  935]
 [1432 1004]
 [1443 1018]
 [1487 1052]
 [1519 1074]
 [1533 1071]
 [1535 1068]
 [1529 1059]
 [1544 1053]
 [1584 1073]
 [1605 1069]
 [1601 1064]
 [1592 1063]
 [1586 1048]
 [1104  708]
 [1008  650]
 [1009  672]
 [1038  700]]
Saved binary mask to: /content/drive/MyDrive/Object_Detection/traffic_sign_object_detection/Dataset/masks/roboflow_train_truth_masks/train7091.jpg
Processing image: train6130_jpg.rf.5fbbe0f5ae277ccd2aee8595ac054388.jpg, dimensions: (1080, 1920)
Polygon points: [[173 961]
 [252 930]
 [285 913]
 [265 913]
 [146 959]
 [146 964]
 [173 961]]
Polygon p

## Paths for testing annotation and output folder

In [ ]:
# Paths for testing annotation and output folder
test_annotation_file = "/content/drive/MyDrive/Object_Detection/traffic_sign_object_detection/Dataset/Roboflow_lane_annotation/test_annotation_coco/_annotations.coco.json"  # Path to your JSON file
test_output_folder = "/content/drive/MyDrive/Object_Detection/traffic_sign_object_detection/Dataset/masks/roboflow_test_truth_masks"  # Path to save binary masks
os.makedirs(test_output_folder, exist_ok=True)

## Test mask creation

In [ ]:
import os
import json
import numpy as np
import cv2

# Load COCO annotations
with open(test_annotation_file, "r") as f:
    coco_data = json.load(f)

# Extract images and annotations
images = {img["id"]: img for img in coco_data["images"]}
annotations = coco_data["annotations"]

# Process each annotation
for image_id, image_info in images.items():
    # Get image details
    image_name = image_info["file_name"]
    height, width = image_info["height"], image_info["width"]
    print(f"Processing image: {image_name}, dimensions: ({height}, {width})")

    # Modify the image name to remove everything from the last underscore to the dot
    cleaned_name = image_name.split('_')[0] + ".jpg"

    # Create an empty binary mask
    binary_mask = np.zeros((height, width), dtype=np.uint8)

    # Find annotations for this image
    image_annotations = [ann for ann in annotations if ann["image_id"] == image_id]

    # Process each annotation for this image
    for annotation in image_annotations:
        segmentation = annotation["segmentation"]

        # Handle polygons
        if isinstance(segmentation, list):  # Polygon format
            for poly in segmentation:
                points = np.array(poly).reshape(-1, 2).astype(np.int32)
                print(f"Polygon points: {points}")
                cv2.fillPoly(binary_mask, [points], color=1)  # Fill the polygon with '1'

    # Save the binary mask
    output_mask_path = os.path.join(test_output_folder, cleaned_name)
    cv2.imwrite(output_mask_path, binary_mask * 255)  # Save as binary JPEG (scaled to 255 for visibility)
    print(f"Saved binary mask to: {output_mask_path}")


## Paths for val annotation and output folder

In [ ]:
# Paths for testing annotation and output folder
val_annotation_file = "/content/drive/MyDrive/Object_Detection/traffic_sign_object_detection/Dataset/Roboflow_lane_annotation/val_annotation_coco/_annotations.coco.json"  # Path to your JSON file
val_output_folder = "/content/drive/MyDrive/Object_Detection/traffic_sign_object_detection/Dataset/masks/roboflow_val_truth_masks"  # Path to save binary masks
os.makedirs(val_output_folder, exist_ok=True)

## Val mask creation

In [ ]:
import os
import json
import numpy as np
import cv2

# Load COCO annotations
with open(val_annotation_file, "r") as f:
    coco_data = json.load(f)

# Extract images and annotations
images = {img["id"]: img for img in coco_data["images"]}
annotations = coco_data["annotations"]

# Process each annotation
for image_id, image_info in images.items():
    # Get image details
    image_name = image_info["file_name"]
    height, width = image_info["height"], image_info["width"]
    print(f"Processing image: {image_name}, dimensions: ({height}, {width})")

    # Modify the image name to remove everything from the last underscore to the dot
    cleaned_name = image_name.split('_')[0] + ".jpg"

    # Create an empty binary mask
    binary_mask = np.zeros((height, width), dtype=np.uint8)

    # Find annotations for this image
    image_annotations = [ann for ann in annotations if ann["image_id"] == image_id]

    # Process each annotation for this image
    for annotation in image_annotations:
        segmentation = annotation["segmentation"]

        # Handle polygons
        if isinstance(segmentation, list):  # Polygon format
            for poly in segmentation:
                points = np.array(poly).reshape(-1, 2).astype(np.int32)
                print(f"Polygon points: {points}")
                cv2.fillPoly(binary_mask, [points], color=1)  # Fill the polygon with '1'

    # Save the binary mask
    output_mask_path = os.path.join(val_output_folder, cleaned_name)
    cv2.imwrite(output_mask_path, binary_mask * 255)  # Save as binary JPEG (scaled to 255 for visibility)
    print(f"Saved binary mask to: {output_mask_path}")


Streaming output truncated to the last 5000 lines.
 [ 764  710]
 [ 753  709]
 [ 767  723]
 [ 776  725]]
Polygon points: [[ 41 908]
 [ 95 880]
 [ 98 874]
 [108 869]
 [ 85 871]
 [ 27 901]
 [ 28 908]
 [ 41 908]]
Polygon points: [[131 861]
 [208 824]
 [191 820]
 [145 825]
 [140 829]
 [146 839]
 [120 856]
 [120 861]
 [131 861]]
Saved binary mask to: /content/drive/MyDrive/Object_Detection/traffic_sign_object_detection/Dataset/masks/roboflow_val_truth_masks/val872.jpg
Processing image: val906_jpg.rf.3fe38b2fd7be72202b07688fb2d812a2.jpg, dimensions: (1080, 1920)
Polygon points: [[646 824]
 [664 821]
 [684 765]
 [669 764]
 [655 788]
 [643 820]
 [646 824]]
Polygon points: [[ 977  639]
 [1029  670]
 [1046  673]
 [1001  641]
 [ 971  627]
 [ 970  631]
 [ 977  639]]
Polygon points: [[1024  668]
 [1036  674]
 [1047  674]
 [1001  642]
 [ 970  627]
 [ 978  639]
 [1024  668]]
Polygon points: [[1138  741]
 [1292  841]
 [1507  974]
 [1638 1060]
 [1666 1075]
 [1699 1074]
 [1137  725]
 [1120  724]
 [1120  

# Check for missing images ( if any )

In this section, we verify the synchronization between **image files** and **mask files** in the training dataset. This step ensures that every image has a corresponding binary mask and vice versa.

### What’s Happening Here?
1. **Extract Filenames**:
   - Extract the base filenames (without extensions) from the images and masks folders.

2. **Compare Sets**:
   - Identify:
     - Images that are missing corresponding masks.
     - Masks that are missing corresponding images.

3. **Display Results**:
   - Print filenames of missing items for manual review.
   - Confirm if the image and mask folders are perfectly synchronized.

In [ ]:
# Get the base filenames (without extensions) for images and masks
image_files = {os.path.splitext(f)[0] for f in os.listdir(train_images) if f.endswith(('.jpg', '.jpeg', '.png'))}
mask_files = {os.path.splitext(f)[0] for f in os.listdir(train_masks) if f.endswith(('.jpg', '.jpeg', '.png'))}

# Check for missing images in masks and vice versa
missing_in_masks = image_files - mask_files
missing_in_images = mask_files - image_files

# Display results
if missing_in_masks:
    print("Images missing masks:")
    for missing in missing_in_masks:
        print(missing + ".jpg")  # Assuming .jpg extension

if missing_in_images:
    print("\nMasks missing corresponding images:")
    for missing in missing_in_images:
        print(missing + ".jpg")  # Assuming .jpg extension

if not missing_in_masks and not missing_in_images:
    print("No missing images or masks. Both folders are in sync.")


No missing images or masks. Both folders are in sync.


# Convert train epochs to array

In [ ]:
import re
# Revisiting the log data parsing to ensure all 100 epochs are included

# Log data was truncated earlier; ensure the full log is parsed here.
log_data = """
Training Epoch 1/100: 100%|██████████| 236/236 [12:20<00:00,  3.14s/it]Epoch [1/100], Train Loss: 2.2334, IoU: 0.0089, Pixel Accuracy: 0.1437
Saving best model...
Training Epoch 2/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [2/100], Train Loss: 2.2207, IoU: 0.0104, Pixel Accuracy: 0.1827
Epoch [2/100], Validation Loss: 2.1958, IoU: 0.0127, Pixel Accuracy: 0.2451
Saving best model...
Training Epoch 3/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [3/100], Train Loss: 2.2111, IoU: 0.0107, Pixel Accuracy: 0.2130
Epoch [3/100], Validation Loss: 2.2193, IoU: 0.0133, Pixel Accuracy: 0.1819
Training Epoch 4/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [4/100], Train Loss: 2.2013, IoU: 0.0112, Pixel Accuracy: 0.2420
Epoch [4/100], Validation Loss: 2.2232, IoU: 0.0120, Pixel Accuracy: 0.1710
Training Epoch 5/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [5/100], Train Loss: 2.1904, IoU: 0.0115, Pixel Accuracy: 0.2713
Epoch [5/100], Validation Loss: 2.2292, IoU: 0.0122, Pixel Accuracy: 0.1566
Training Epoch 6/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [6/100], Train Loss: 2.1789, IoU: 0.0118, Pixel Accuracy: 0.3021
Epoch [6/100], Validation Loss: 2.2079, IoU: 0.0131, Pixel Accuracy: 0.2123
Training Epoch 7/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [7/100], Train Loss: 2.1671, IoU: 0.0122, Pixel Accuracy: 0.3342
Epoch [7/100], Validation Loss: 2.2061, IoU: 0.0124, Pixel Accuracy: 0.2183
Training Epoch 8/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [8/100], Train Loss: 2.1545, IoU: 0.0132, Pixel Accuracy: 0.3684
Epoch [8/100], Validation Loss: 2.1885, IoU: 0.0136, Pixel Accuracy: 0.2715
Saving best model...
Training Epoch 9/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [9/100], Train Loss: 2.1413, IoU: 0.0141, Pixel Accuracy: 0.4027
Epoch [9/100], Validation Loss: 2.1724, IoU: 0.0156, Pixel Accuracy: 0.3083
Saving best model...
Training Epoch 10/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [10/100], Train Loss: 2.1283, IoU: 0.0145, Pixel Accuracy: 0.4372
Epoch [10/100], Validation Loss: 2.1250, IoU: 0.0160, Pixel Accuracy: 0.4318
Saving best model...
Training Epoch 11/100: 100%|██████████| 236/236 [01:00<00:00,  3.91it/s]Epoch [11/100], Train Loss: 2.1151, IoU: 0.0147, Pixel Accuracy: 0.4709
Epoch [11/100], Validation Loss: 2.0789, IoU: 0.0170, Pixel Accuracy: 0.5533
Saving best model...
Training Epoch 12/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [12/100], Train Loss: 2.1014, IoU: 0.0166, Pixel Accuracy: 0.5060
Epoch [12/100], Validation Loss: 2.1010, IoU: 0.0182, Pixel Accuracy: 0.5110
Training Epoch 13/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [13/100], Train Loss: 2.0884, IoU: 0.0168, Pixel Accuracy: 0.5382
Epoch [13/100], Validation Loss: 2.0894, IoU: 0.0173, Pixel Accuracy: 0.5269
Training Epoch 14/100: 100%|██████████| 236/236 [01:00<00:00,  3.92it/s]Epoch [14/100], Train Loss: 2.0750, IoU: 0.0187, Pixel Accuracy: 0.5717
Epoch [14/100], Validation Loss: 2.1243, IoU: 0.0158, Pixel Accuracy: 0.4324
Training Epoch 15/100: 100%|██████████| 236/236 [01:00<00:00,  3.92it/s]Epoch [15/100], Train Loss: 2.0630, IoU: 0.0191, Pixel Accuracy: 0.6013
Epoch [15/100], Validation Loss: 2.0664, IoU: 0.0204, Pixel Accuracy: 0.5755
Saving best model...
Training Epoch 16/100: 100%|██████████| 236/236 [01:00<00:00,  3.91it/s]Epoch [16/100], Train Loss: 2.0507, IoU: 0.0210, Pixel Accuracy: 0.6317
Epoch [16/100], Validation Loss: 2.1124, IoU: 0.0171, Pixel Accuracy: 0.4618
Training Epoch 17/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [17/100], Train Loss: 2.0390, IoU: 0.0222, Pixel Accuracy: 0.6601
Epoch [17/100], Validation Loss: 2.0545, IoU: 0.0227, Pixel Accuracy: 0.6156
Saving best model...
Training Epoch 18/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [18/100], Train Loss: 2.0272, IoU: 0.0251, Pixel Accuracy: 0.6877
Epoch [18/100], Validation Loss: 2.0218, IoU: 0.0309, Pixel Accuracy: 0.6991
Saving best model...
Training Epoch 19/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [19/100], Train Loss: 2.0164, IoU: 0.0268, Pixel Accuracy: 0.7136
Epoch [19/100], Validation Loss: 2.0294, IoU: 0.0229, Pixel Accuracy: 0.6349
Training Epoch 20/100: 100%|██████████| 236/236 [01:01<00:00,  3.83it/s]Epoch [20/100], Train Loss: 2.0065, IoU: 0.0286, Pixel Accuracy: 0.7380
Epoch [20/100], Validation Loss: 2.0285, IoU: 0.0238, Pixel Accuracy: 0.6658
Training Epoch 21/100: 100%|██████████| 236/236 [01:01<00:00,  3.86it/s]Epoch [21/100], Train Loss: 1.9970, IoU: 0.0298, Pixel Accuracy: 0.7592
Epoch [21/100], Validation Loss: 1.9669, IoU: 0.0508, Pixel Accuracy: 0.8183
Saving best model...
Training Epoch 22/100: 100%|██████████| 236/236 [01:01<00:00,  3.87it/s]Epoch [22/100], Train Loss: 1.9876, IoU: 0.0341, Pixel Accuracy: 0.7819
Epoch [22/100], Validation Loss: 1.9552, IoU: 0.0627, Pixel Accuracy: 0.8494
Saving best model...
Training Epoch 23/100: 100%|██████████| 236/236 [01:01<00:00,  3.85it/s]Epoch [23/100], Train Loss: 1.9789, IoU: 0.0377, Pixel Accuracy: 0.8018
Epoch [23/100], Validation Loss: 1.9754, IoU: 0.0544, Pixel Accuracy: 0.7964
Training Epoch 24/100: 100%|██████████| 236/236 [01:01<00:00,  3.87it/s]Epoch [24/100], Train Loss: 1.9707, IoU: 0.0413, Pixel Accuracy: 0.8206
Epoch [24/100], Validation Loss: 1.9516, IoU: 0.0648, Pixel Accuracy: 0.8631
Saving best model...
Training Epoch 25/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]
Epoch [25/100], Train Loss: 1.9634, IoU: 0.0446, Pixel Accuracy: 0.8363
Epoch [25/100], Validation Loss: 1.9425, IoU: 0.0722, Pixel Accuracy: 0.8951
Saving best model...
Training Epoch 26/100: 100%|██████████| 236/236 [01:01<00:00,  3.82it/s]Epoch [26/100], Train Loss: 1.9563, IoU: 0.0510, Pixel Accuracy: 0.8541
Epoch [26/100], Validation Loss: 1.9493, IoU: 0.0673, Pixel Accuracy: 0.8638
Training Epoch 27/100: 100%|██████████| 236/236 [01:01<00:00,  3.85it/s]Epoch [27/100], Train Loss: 1.9502, IoU: 0.0529, Pixel Accuracy: 0.8686
Epoch [27/100], Validation Loss: 1.9873, IoU: 0.0384, Pixel Accuracy: 0.7609
Training Epoch 28/100: 100%|██████████| 236/236 [01:01<00:00,  3.85it/s]Epoch [28/100], Train Loss: 1.9445, IoU: 0.0555, Pixel Accuracy: 0.8802
Epoch [28/100], Validation Loss: 1.9475, IoU: 0.0809, Pixel Accuracy: 0.8621
Training Epoch 29/100: 100%|██████████| 236/236 [01:00<00:00,  3.87it/s]Epoch [29/100], Train Loss: 1.9388, IoU: 0.0639, Pixel Accuracy: 0.8932
Epoch [29/100], Validation Loss: 1.9542, IoU: 0.0609, Pixel Accuracy: 0.8436
Training Epoch 30/100: 100%|██████████| 236/236 [01:01<00:00,  3.84it/s]Epoch [30/100], Train Loss: 1.9334, IoU: 0.0742, Pixel Accuracy: 0.9054
Epoch [30/100], Validation Loss: 1.9425, IoU: 0.0805, Pixel Accuracy: 0.8745
Saving best model...
Training Epoch 31/100: 100%|██████████| 236/236 [01:01<00:00,  3.85it/s]Epoch [31/100], Train Loss: 1.9289, IoU: 0.0804, Pixel Accuracy: 0.9152
Epoch [31/100], Validation Loss: 1.9296, IoU: 0.0859, Pixel Accuracy: 0.9187
Saving best model...
Training Epoch 32/100: 100%|██████████| 236/236 [01:01<00:00,  3.84it/s]
Epoch [32/100], Train Loss: 1.9260, IoU: 0.0723, Pixel Accuracy: 0.9222
Epoch [32/100], Validation Loss: 1.9111, IoU: 0.1067, Pixel Accuracy: 0.9471
Saving best model...
Training Epoch 33/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]
Epoch [33/100], Train Loss: 1.9212, IoU: 0.0922, Pixel Accuracy: 0.9327
Epoch [33/100], Validation Loss: 1.9166, IoU: 0.1036, Pixel Accuracy: 0.9344
Training Epoch 34/100: 100%|██████████| 236/236 [01:01<00:00,  3.87it/s]Epoch [34/100], Train Loss: 1.9172, IoU: 0.1116, Pixel Accuracy: 0.9417
Epoch [34/100], Validation Loss: 1.9155, IoU: 0.1240, Pixel Accuracy: 0.9414
Training Epoch 35/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [35/100], Train Loss: 1.9138, IoU: 0.1290, Pixel Accuracy: 0.9497
Epoch [35/100], Validation Loss: 1.9090, IoU: 0.1412, Pixel Accuracy: 0.9516
Saving best model...
Training Epoch 36/100: 100%|██████████| 236/236 [01:01<00:00,  3.82it/s]Epoch [36/100], Train Loss: 1.9108, IoU: 0.1442, Pixel Accuracy: 0.9559
Epoch [36/100], Validation Loss: 1.9055, IoU: 0.1643, Pixel Accuracy: 0.9589
Saving best model...
Training Epoch 37/100: 100%|██████████| 236/236 [01:00<00:00,  3.87it/s]Epoch [37/100], Train Loss: 1.9083, IoU: 0.1591, Pixel Accuracy: 0.9612
Epoch [37/100], Validation Loss: 1.9032, IoU: 0.1841, Pixel Accuracy: 0.9655
Saving best model...
Training Epoch 38/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [38/100], Train Loss: 1.9064, IoU: 0.1725, Pixel Accuracy: 0.9650
Epoch [38/100], Validation Loss: 1.9022, IoU: 0.1974, Pixel Accuracy: 0.9680
Saving best model...
Training Epoch 39/100: 100%|██████████| 236/236 [01:00<00:00,  3.93it/s]Epoch [39/100], Train Loss: 1.9045, IoU: 0.1883, Pixel Accuracy: 0.9691
Epoch [39/100], Validation Loss: 1.9026, IoU: 0.1879, Pixel Accuracy: 0.9663
Training Epoch 40/100: 100%|██████████| 236/236 [01:00<00:00,  3.87it/s]Epoch [40/100], Train Loss: 1.9031, IoU: 0.2042, Pixel Accuracy: 0.9725
Epoch [40/100], Validation Loss: 1.8990, IoU: 0.2074, Pixel Accuracy: 0.9745
Saving best model...
Training Epoch 41/100: 100%|██████████| 236/236 [01:01<00:00,  3.83it/s]Epoch [41/100], Train Loss: 1.9018, IoU: 0.2174, Pixel Accuracy: 0.9753
Epoch [41/100], Validation Loss: 1.8985, IoU: 0.2383, Pixel Accuracy: 0.9767
Saving best model...
Training Epoch 42/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [42/100], Train Loss: 1.9009, IoU: 0.2275, Pixel Accuracy: 0.9771
Epoch [42/100], Validation Loss: 1.8980, IoU: 0.2360, Pixel Accuracy: 0.9783
Saving best model...
Training Epoch 43/100: 100%|██████████| 236/236 [01:01<00:00,  3.84it/s]Epoch [43/100], Train Loss: 1.9002, IoU: 0.2339, Pixel Accuracy: 0.9780
Epoch [43/100], Validation Loss: 1.8984, IoU: 0.2408, Pixel Accuracy: 0.9789
Training Epoch 44/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [44/100], Train Loss: 1.8998, IoU: 0.2386, Pixel Accuracy: 0.9792
Epoch [44/100], Validation Loss: 1.8968, IoU: 0.2458, Pixel Accuracy: 0.9798
Saving best model...
Training Epoch 45/100: 100%|██████████| 236/236 [01:01<00:00,  3.85it/s]Epoch [45/100], Train Loss: 1.8996, IoU: 0.2378, Pixel Accuracy: 0.9796
Epoch [45/100], Validation Loss: 1.8970, IoU: 0.2410, Pixel Accuracy: 0.9799
Training Epoch 46/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [46/100], Train Loss: 1.8995, IoU: 0.2409, Pixel Accuracy: 0.9798
Epoch [46/100], Validation Loss: 1.8983, IoU: 0.2004, Pixel Accuracy: 0.9753
Training Epoch 47/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [47/100], Train Loss: 1.8992, IoU: 0.2433, Pixel Accuracy: 0.9805
Epoch [47/100], Validation Loss: 1.8965, IoU: 0.2376, Pixel Accuracy: 0.9806
Saving best model...
Training Epoch 48/100: 100%|██████████| 236/236 [01:01<00:00,  3.86it/s]Epoch [48/100], Train Loss: 1.8990, IoU: 0.2511, Pixel Accuracy: 0.9807
Epoch [48/100], Validation Loss: 1.8964, IoU: 0.2639, Pixel Accuracy: 0.9807
Saving best model...
Training Epoch 49/100: 100%|██████████| 236/236 [01:01<00:00,  3.86it/s]Epoch [49/100], Train Loss: 1.8986, IoU: 0.2749, Pixel Accuracy: 0.9821
Epoch [49/100], Validation Loss: 1.8950, IoU: 0.3053, Pixel Accuracy: 0.9842
Saving best model...
Training Epoch 50/100: 100%|██████████| 236/236 [01:01<00:00,  3.83it/s]Epoch [50/100], Train Loss: 1.8980, IoU: 0.2972, Pixel Accuracy: 0.9837
Epoch [50/100], Validation Loss: 1.8956, IoU: 0.2884, Pixel Accuracy: 0.9823
Training Epoch 51/100: 100%|██████████| 236/236 [01:01<00:00,  3.86it/s]Epoch [51/100], Train Loss: 1.8977, IoU: 0.3032, Pixel Accuracy: 0.9841
Epoch [51/100], Validation Loss: 1.8948, IoU: 0.3055, Pixel Accuracy: 0.9846
Saving best model...
Training Epoch 52/100: 100%|██████████| 236/236 [01:00<00:00,  3.87it/s]Epoch [52/100], Train Loss: 1.8975, IoU: 0.3139, Pixel Accuracy: 0.9846
Epoch [52/100], Validation Loss: 1.8946, IoU: 0.3279, Pixel Accuracy: 0.9847
Saving best model...
Training Epoch 53/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [53/100], Train Loss: 1.8973, IoU: 0.3277, Pixel Accuracy: 0.9848
Epoch [53/100], Validation Loss: 1.8945, IoU: 0.3281, Pixel Accuracy: 0.9846
Saving best model...
Training Epoch 54/100: 100%|██████████| 236/236 [01:00<00:00,  3.87it/s]Epoch [54/100], Train Loss: 1.8971, IoU: 0.3380, Pixel Accuracy: 0.9849
Epoch [54/100], Validation Loss: 1.8944, IoU: 0.3401, Pixel Accuracy: 0.9849
Saving best model...
Training Epoch 55/100: 100%|██████████| 236/236 [01:00<00:00,  3.93it/s]Epoch [55/100], Train Loss: 1.8971, IoU: 0.3442, Pixel Accuracy: 0.9851
Epoch [55/100], Validation Loss: 1.8943, IoU: 0.3441, Pixel Accuracy: 0.9852
Saving best model...
Training Epoch 56/100: 100%|██████████| 236/236 [01:00<00:00,  3.91it/s]Epoch [56/100], Train Loss: 1.8970, IoU: 0.3488, Pixel Accuracy: 0.9852
Epoch [56/100], Validation Loss: 1.8942, IoU: 0.3574, Pixel Accuracy: 0.9853
Saving best model...
Training Epoch 57/100: 100%|██████████| 236/236 [01:00<00:00,  3.91it/s]Epoch [57/100], Train Loss: 1.8969, IoU: 0.3554, Pixel Accuracy: 0.9853
Epoch [57/100], Validation Loss: 1.8942, IoU: 0.3540, Pixel Accuracy: 0.9852
Saving best model...
Training Epoch 58/100: 100%|██████████| 236/236 [01:00<00:00,  3.92it/s]Epoch [58/100], Train Loss: 1.8969, IoU: 0.3583, Pixel Accuracy: 0.9854
Epoch [58/100], Validation Loss: 1.8942, IoU: 0.3570, Pixel Accuracy: 0.9856
Training Epoch 59/100: 100%|██████████| 236/236 [01:00<00:00,  3.92it/s]Epoch [59/100], Train Loss: 1.8969, IoU: 0.3655, Pixel Accuracy: 0.9855
Epoch [59/100], Validation Loss: 1.8941, IoU: 0.3657, Pixel Accuracy: 0.9855
Saving best model...
Training Epoch 60/100: 100%|██████████| 236/236 [01:00<00:00,  3.93it/s]Epoch [60/100], Train Loss: 1.8952, IoU: 0.4775, Pixel Accuracy: 0.9901
Epoch [60/100], Validation Loss: 1.8920, IoU: 0.4916, Pixel Accuracy: 0.9917
Saving best model...
Training Epoch 61/100: 100%|██████████| 236/236 [00:59<00:00,  3.94it/s]Epoch [61/100], Train Loss: 1.8943, IoU: 0.5458, Pixel Accuracy: 0.9922
Epoch [61/100], Validation Loss: 1.8914, IoU: 0.5576, Pixel Accuracy: 0.9925
Saving best model...
Training Epoch 62/100: 100%|██████████| 236/236 [00:59<00:00,  3.98it/s]Epoch [62/100], Train Loss: 1.8941, IoU: 0.5790, Pixel Accuracy: 0.9927
Epoch [62/100], Validation Loss: 1.8912, IoU: 0.5850, Pixel Accuracy: 0.9928
Saving best model...
Training Epoch 63/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [63/100], Train Loss: 1.8939, IoU: 0.5923, Pixel Accuracy: 0.9928
Epoch [63/100], Validation Loss: 1.8912, IoU: 0.5856, Pixel Accuracy: 0.9929
Saving best model...
Training Epoch 64/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [64/100], Train Loss: 1.8938, IoU: 0.6102, Pixel Accuracy: 0.9931
Epoch [64/100], Validation Loss: 1.8910, IoU: 0.6129, Pixel Accuracy: 0.9931
Saving best model...
Training Epoch 65/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [65/100], Train Loss: 1.8938, IoU: 0.6179, Pixel Accuracy: 0.9931
Epoch [65/100], Validation Loss: 1.8909, IoU: 0.6221, Pixel Accuracy: 0.9931
Saving best model...
Training Epoch 66/100: 100%|██████████| 236/236 [00:59<00:00,  3.94it/s]Epoch [66/100], Train Loss: 1.8937, IoU: 0.6257, Pixel Accuracy: 0.9932
Epoch [66/100], Validation Loss: 1.8909, IoU: 0.6297, Pixel Accuracy: 0.9933
Saving best model...
Training Epoch 67/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [67/100], Train Loss: 1.8936, IoU: 0.6326, Pixel Accuracy: 0.9933
Epoch [67/100], Validation Loss: 1.8910, IoU: 0.6312, Pixel Accuracy: 0.9935
Training Epoch 68/100: 100%|██████████| 236/236 [01:00<00:00,  3.91it/s]Epoch [68/100], Train Loss: 1.8937, IoU: 0.6333, Pixel Accuracy: 0.9933
Epoch [68/100], Validation Loss: 1.8909, IoU: 0.6234, Pixel Accuracy: 0.9934
Saving best model...
Training Epoch 69/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [69/100], Train Loss: 1.8936, IoU: 0.6426, Pixel Accuracy: 0.9934
Epoch [69/100], Validation Loss: 1.8908, IoU: 0.6385, Pixel Accuracy: 0.9934
Saving best model...
Training Epoch 70/100: 100%|██████████| 236/236 [01:01<00:00,  3.86it/s]Epoch [70/100], Train Loss: 1.8935, IoU: 0.6484, Pixel Accuracy: 0.9934
Epoch [70/100], Validation Loss: 1.8908, IoU: 0.6388, Pixel Accuracy: 0.9935
Saving best model...
Training Epoch 71/100: 100%|██████████| 236/236 [01:00<00:00,  3.92it/s]Epoch [71/100], Train Loss: 1.8935, IoU: 0.6530, Pixel Accuracy: 0.9935
Epoch [71/100], Validation Loss: 1.8908, IoU: 0.6406, Pixel Accuracy: 0.9935
Saving best model...
Training Epoch 72/100: 100%|██████████| 236/236 [01:00<00:00,  3.92it/s]Epoch [72/100], Train Loss: 1.8934, IoU: 0.6577, Pixel Accuracy: 0.9935
Epoch [72/100], Validation Loss: 1.8908, IoU: 0.6147, Pixel Accuracy: 0.9932
Training Epoch 73/100: 100%|██████████| 236/236 [01:00<00:00,  3.91it/s]Epoch [73/100], Train Loss: 1.8934, IoU: 0.6595, Pixel Accuracy: 0.9936
Epoch [73/100], Validation Loss: 1.8908, IoU: 0.6568, Pixel Accuracy: 0.9936
Training Epoch 74/100: 100%|██████████| 236/236 [00:59<00:00,  3.96it/s]Epoch [74/100], Train Loss: 1.8934, IoU: 0.6659, Pixel Accuracy: 0.9937
Epoch [74/100], Validation Loss: 1.8907, IoU: 0.6517, Pixel Accuracy: 0.9935
Saving best model...
Training Epoch 75/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [75/100], Train Loss: 1.8933, IoU: 0.6680, Pixel Accuracy: 0.9937
Epoch [75/100], Validation Loss: 1.8907, IoU: 0.6476, Pixel Accuracy: 0.9935
Saving best model...
Training Epoch 76/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [76/100], Train Loss: 1.8934, IoU: 0.6697, Pixel Accuracy: 0.9937
Epoch [76/100], Validation Loss: 1.8907, IoU: 0.6594, Pixel Accuracy: 0.9936
Saving best model...
Training Epoch 77/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [77/100], Train Loss: 1.8933, IoU: 0.6702, Pixel Accuracy: 0.9937
Epoch [77/100], Validation Loss: 1.8907, IoU: 0.6337, Pixel Accuracy: 0.9931
Training Epoch 78/100: 100%|██████████| 236/236 [01:01<00:00,  3.86it/s]Epoch [78/100], Train Loss: 1.8934, IoU: 0.6711, Pixel Accuracy: 0.9937
Epoch [78/100], Validation Loss: 1.8907, IoU: 0.6286, Pixel Accuracy: 0.9933
Training Epoch 79/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [79/100], Train Loss: 1.8934, IoU: 0.6739, Pixel Accuracy: 0.9937
Epoch [79/100], Validation Loss: 1.8908, IoU: 0.6512, Pixel Accuracy: 0.9935
Training Epoch 80/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]
Epoch [80/100], Train Loss: 1.8933, IoU: 0.6772, Pixel Accuracy: 0.9938
Epoch [80/100], Validation Loss: 1.8906, IoU: 0.6639, Pixel Accuracy: 0.9935
Saving best model...
Training Epoch 81/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [81/100], Train Loss: 1.8934, IoU: 0.6782, Pixel Accuracy: 0.9938
Epoch [81/100], Validation Loss: 1.8907, IoU: 0.6378, Pixel Accuracy: 0.9932
Training Epoch 82/100: 100%|██████████| 236/236 [01:00<00:00,  3.87it/s]Epoch [82/100], Train Loss: 1.8933, IoU: 0.6835, Pixel Accuracy: 0.9938
Epoch [82/100], Validation Loss: 1.8906, IoU: 0.6796, Pixel Accuracy: 0.9938
Training Epoch 83/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [83/100], Train Loss: 1.8933, IoU: 0.6853, Pixel Accuracy: 0.9939
Epoch [83/100], Validation Loss: 1.8906, IoU: 0.6692, Pixel Accuracy: 0.9937
Training Epoch 84/100: 100%|██████████| 236/236 [00:59<00:00,  3.93it/s]Epoch [84/100], Train Loss: 1.8932, IoU: 0.6894, Pixel Accuracy: 0.9939
Epoch [84/100], Validation Loss: 1.8906, IoU: 0.6765, Pixel Accuracy: 0.9937
Training Epoch 85/100: 100%|██████████| 236/236 [01:00<00:00,  3.92it/s]Epoch [85/100], Train Loss: 1.8933, IoU: 0.6920, Pixel Accuracy: 0.9939
Epoch [85/100], Validation Loss: 1.8906, IoU: 0.6736, Pixel Accuracy: 0.9938
Saving best model...
Training Epoch 86/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [86/100], Train Loss: 1.8933, IoU: 0.6896, Pixel Accuracy: 0.9939
Epoch [86/100], Validation Loss: 1.8905, IoU: 0.6740, Pixel Accuracy: 0.9937
Saving best model...
Training Epoch 87/100: 100%|██████████| 236/236 [01:00<00:00,  3.92it/s]Epoch [87/100], Train Loss: 1.8932, IoU: 0.6950, Pixel Accuracy: 0.9940
Epoch [87/100], Validation Loss: 1.8906, IoU: 0.6584, Pixel Accuracy: 0.9936
Training Epoch 88/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [88/100], Train Loss: 1.8932, IoU: 0.6951, Pixel Accuracy: 0.9939
Epoch [88/100], Validation Loss: 1.8906, IoU: 0.6580, Pixel Accuracy: 0.9936
Training Epoch 89/100: 100%|██████████| 236/236 [01:00<00:00,  3.93it/s]Epoch [89/100], Train Loss: 1.8933, IoU: 0.6985, Pixel Accuracy: 0.9940
Epoch [89/100], Validation Loss: 1.8905, IoU: 0.6610, Pixel Accuracy: 0.9936
Training Epoch 90/100: 100%|██████████| 236/236 [01:01<00:00,  3.87it/s]Epoch [90/100], Train Loss: 1.8932, IoU: 0.6984, Pixel Accuracy: 0.9940
Epoch [90/100], Validation Loss: 1.8905, IoU: 0.6681, Pixel Accuracy: 0.9936
Training Epoch 91/100: 100%|██████████| 236/236 [01:00<00:00,  3.91it/s]Epoch [91/100], Train Loss: 1.8931, IoU: 0.7027, Pixel Accuracy: 0.9940
Epoch [91/100], Validation Loss: 1.8907, IoU: 0.6578, Pixel Accuracy: 0.9938
Training Epoch 92/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [92/100], Train Loss: 1.8932, IoU: 0.6982, Pixel Accuracy: 0.9940
Epoch [92/100], Validation Loss: 1.8905, IoU: 0.6859, Pixel Accuracy: 0.9938
Saving best model...
Training Epoch 93/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [93/100], Train Loss: 1.8932, IoU: 0.7042, Pixel Accuracy: 0.9941
Epoch [93/100], Validation Loss: 1.8905, IoU: 0.6634, Pixel Accuracy: 0.9935
Training Epoch 94/100: 100%|██████████| 236/236 [01:00<00:00,  3.88it/s]Epoch [94/100], Train Loss: 1.8932, IoU: 0.7009, Pixel Accuracy: 0.9940
Epoch [94/100], Validation Loss: 1.8905, IoU: 0.6798, Pixel Accuracy: 0.9938
Saving best model...
Training Epoch 95/100: 100%|██████████| 236/236 [01:01<00:00,  3.84it/s]Epoch [95/100], Train Loss: 1.8930, IoU: 0.7062, Pixel Accuracy: 0.9941
Epoch [95/100], Validation Loss: 1.8905, IoU: 0.6836, Pixel Accuracy: 0.9938
Saving best model...
Training Epoch 96/100: 100%|██████████| 236/236 [01:00<00:00,  3.90it/s]Epoch [96/100], Train Loss: 1.8932, IoU: 0.7067, Pixel Accuracy: 0.9941
Epoch [96/100], Validation Loss: 1.8905, IoU: 0.6868, Pixel Accuracy: 0.9940
Training Epoch 97/100: 100%|██████████| 236/236 [01:00<00:00,  3.91it/s]Epoch [97/100], Train Loss: 1.8931, IoU: 0.7063, Pixel Accuracy: 0.9941
Epoch [97/100], Validation Loss: 1.8905, IoU: 0.6780, Pixel Accuracy: 0.9938
Training Epoch 98/100: 100%|██████████| 236/236 [01:00<00:00,  3.89it/s]Epoch [98/100], Train Loss: 1.8931, IoU: 0.7092, Pixel Accuracy: 0.9941
Epoch [98/100], Validation Loss: 1.8905, IoU: 0.6759, Pixel Accuracy: 0.9938
Training Epoch 99/100: 100%|██████████| 236/236 [01:02<00:00,  3.79it/s]Epoch [99/100], Train Loss: 1.8931, IoU: 0.7126, Pixel Accuracy: 0.9941
Epoch [99/100], Validation Loss: 1.8905, IoU: 0.6775, Pixel Accuracy: 0.9937
Training Epoch 100/100: 100%|██████████| 236/236 [01:00<00:00,  3.87it/s]Epoch [100/100], Train Loss: 1.8931, IoU: 0.7116, Pixel Accuracy: 0.9941
Epoch [100/100], Validation Loss: 1.8904, IoU: 0.6879, Pixel Accuracy: 0.9938
Saving best model...
"""

# Adjust parsing logic to ensure all epochs are captured
train_pattern = r"Epoch\s\[(\d+)/100\],\sTrain\sLoss:\s([\d.]+),\sIoU:\s([\d.]+),\sPixel\sAccuracy:\s([\d.]+)"
val_pattern = r"Epoch\s\[\d+/100\],\sValidation\sLoss:\s([\d.]+),\sIoU:\s([\d.]+),\sPixel\sAccuracy:\s([\d.]+)"

# Re-extract train and validation metrics from the log
train_matches = re.findall(train_pattern, log_data)
val_matches = re.findall(val_pattern, log_data)

# Build full results array
results = []
for train, val in zip(train_matches, val_matches):
    epoch = int(train[0])
    results.append({
        "epoch": epoch,
        "train_loss": float(train[1]),
        "train_iou": float(train[2]),
        "train_accuracy": float(train[3]),
        "val_loss": float(val[0]),
        "val_iou": float(val[1]),
        "val_accuracy": float(val[2]),
    })

# Displaying the full array of results
results


[{'epoch': 1,
  'train_loss': 2.2334,
  'train_iou': 0.0089,
  'train_accuracy': 0.1437,
  'val_loss': 2.1958,
  'val_iou': 0.0127,
  'val_accuracy': 0.2451},
 {'epoch': 2,
  'train_loss': 2.2207,
  'train_iou': 0.0104,
  'train_accuracy': 0.1827,
  'val_loss': 2.2193,
  'val_iou': 0.0133,
  'val_accuracy': 0.1819},
 {'epoch': 3,
  'train_loss': 2.2111,
  'train_iou': 0.0107,
  'train_accuracy': 0.213,
  'val_loss': 2.2232,
  'val_iou': 0.012,
  'val_accuracy': 0.171},
 {'epoch': 4,
  'train_loss': 2.2013,
  'train_iou': 0.0112,
  'train_accuracy': 0.242,
  'val_loss': 2.2292,
  'val_iou': 0.0122,
  'val_accuracy': 0.1566},
 {'epoch': 5,
  'train_loss': 2.1904,
  'train_iou': 0.0115,
  'train_accuracy': 0.2713,
  'val_loss': 2.2079,
  'val_iou': 0.0131,
  'val_accuracy': 0.2123},
 {'epoch': 6,
  'train_loss': 2.1789,
  'train_iou': 0.0118,
  'train_accuracy': 0.3021,
  'val_loss': 2.2061,
  'val_iou': 0.0124,
  'val_accuracy': 0.2183},
 {'epoch': 7,
  'train_loss': 2.1671,
  'train_iou